# Pauli Algebra 02: Measurements and Sparse States

This notebook focuses on evaluating expectation values without expanding every operator into a dense matrix.


## What you will learn

1. How to measure a `PauliString` or `PauliSum` on a dense state vector.
2. How to represent sparse pure states with `SparseKet`.
3. How to measure dense density matrices.
4. How basis expectation vectors are prepared for Lie-basis dynamics.


In [1]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys
import types


def _ensure_ipython_display_if_missing():
    try:
        import IPython.display  # noqa: F401
        return
    except ModuleNotFoundError:
        pass

    ipython_module = types.ModuleType("IPython")
    display_module = types.ModuleType("IPython.display")
    display_module.display = lambda *args, **kwargs: None
    display_module.Math = lambda *args, **kwargs: None
    ipython_module.display = display_module
    ipython_module.get_ipython = lambda: None
    ipython_module.version_info = (0, 0)
    sys.modules.setdefault("IPython", ipython_module)
    sys.modules.setdefault("IPython.display", display_module)


def _import_pauli_algebra():
    _ensure_ipython_display_if_missing()
    return importlib.import_module("quantum_simulation.pauli_algebra")


try:
    pa = _import_pauli_algebra()
except ModuleNotFoundError as exc:
    if exc.name != "quantum_simulation":
        raise
    repo_dir = Path("quantum-simulation")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/ToelUl/quantum-simulation.git"], check=True)
    target = Path("quantum_simulation")
    if not target.exists():
        shutil.copytree(repo_dir / "quantum_simulation", target)
    pa = _import_pauli_algebra()

print("Loaded quantum_simulation.pauli_algebra from", Path(pa.__file__).parent)


Loaded quantum_simulation.pauli_algebra from /mnt/d/phd_research_projects/quantum-simulation-main/quantum_simulation/pauli_algebra


In [2]:
import numpy as np
import torch

from quantum_simulation.pauli_algebra import (
    PauliString,
    PauliSum,
    SparseKet,
    pauli_string_from_sites,
    measure_pauli_sum_state_vector,
    measure_pauli_sum_density_matrix,
    basis_expectations_for_sparse_ket,
    basis_expectations_for_state_vector,
    basis_expectations_for_density_matrix,
)

np.set_printoptions(precision=4, suppress=True)
device = "cpu"
dtype = torch.complex128
print(f"Using device={device}, dtype={dtype}")


Using device=cpu, dtype=torch.complex128


## Build a small observable

The examples use three qubits and the observable `Z_0 + X_0 X_1`. The labels still follow the module convention: site 0 is the first character and the least significant bit.


In [3]:
n = 3
z0 = pauli_string_from_sites("Z", [0], lattice_length=n)
xx01 = pauli_string_from_sites("XX", [0, 1], lattice_length=n, coefficient=0.5)
observable = PauliSum([z0, xx01]).simplify()

print("observable =", observable)
print("term labels =", [term.label for term in observable.terms])


observable = ZII + ((0.5+0j))*XXI
term labels = ['ZII', 'XXI']


## Dense state-vector measurement

The vectorized measurement path acts directly on computational-basis indices. For small systems, it is useful to cross-check against a dense matrix expectation value.


In [4]:
psi = np.zeros(2**n, dtype=complex)
psi[0b000] = 1 / np.sqrt(2)
psi[0b111] = 1 / np.sqrt(2)

value_vectorized = measure_pauli_sum_state_vector(observable, psi, device=device)
value_dense = np.vdot(psi, observable.to_matrix() @ psi)

print("vectorized <O> =", value_vectorized)
print("dense      <O> =", value_dense)
assert np.allclose(value_vectorized, value_dense)


vectorized <O> = 0j
dense      <O> = 0j


## SparseKet measurement

`SparseKet` stores only occupied basis states and amplitudes. This is useful when the state is a basis state or a small superposition.


In [5]:
sparse_ghz = SparseKet(
    n=n,
    indices=np.array([0b000, 0b111], dtype=np.uint64),
    amps=torch.tensor([1.0, 1.0], dtype=dtype),
    device=device,
).normalize_()

value_sparse = sparse_ghz.measure_pauli(observable)
print("SparseKet norm =", sparse_ghz.norm())
print("SparseKet <O> =", value_sparse)
assert np.allclose(value_sparse, value_dense)


SparseKet norm = 0.9999999999999999
SparseKet <O> = 0j


## Applying a PauliSum to a SparseKet

Applying a symbolic operator flips the relevant bit masks and accumulates the Pauli phases, again without creating a full state vector.


In [6]:
x0 = pauli_string_from_sites("X", [0], lattice_length=n)
ket_000 = SparseKet.basis(n, 0b000, device=device)
flipped = ket_000.apply_pauli_sum_vectorized(PauliSum([x0]))

print("X_0 |000> indices =", flipped.indices.tolist())
print("amplitudes =", flipped.amps.cpu().numpy())
assert flipped.indices.tolist() == [0b001]
assert np.allclose(flipped.amps.cpu().numpy(), np.array([1.0 + 0.0j]))


X_0 |000> indices = [1]
amplitudes = [1.+0.j]


## Dense density-matrix measurement

For mixed states, use `measure_pauli_sum_density_matrix`. It evaluates `Tr(O rho)` from the density matrix entries and the Pauli bit masks.


In [7]:
rho = np.zeros((2**n, 2**n), dtype=complex)
rho[0b000, 0b000] = 0.7
rho[0b111, 0b111] = 0.3

value_rho = measure_pauli_sum_density_matrix(observable, rho, device=device)
value_rho_dense = np.trace(observable.to_matrix() @ rho)

print("vectorized Tr(O rho) =", value_rho)
print("dense      Tr(O rho) =", value_rho_dense)
assert np.allclose(value_rho, value_rho_dense)


vectorized Tr(O rho) = (0.39999999999999997+0j)
dense      Tr(O rho) = (0.39999999999999997+0j)


## Basis expectation vectors

The three `basis_expectations_for_*` helpers return the vector of moments used later by the Lie-closure dynamics API.


In [8]:
basis = [PauliSum([z0]), PauliSum([xx01])]

mu_sparse = basis_expectations_for_sparse_ket(basis, sparse_ghz, device=device)
mu_vector = basis_expectations_for_state_vector(basis, psi, device=device)
mu_rho = basis_expectations_for_density_matrix(basis, rho, device=device)

print("mu from sparse ket :", mu_sparse.cpu().numpy())
print("mu from state vector:", mu_vector.cpu().numpy())
print("mu from rho         :", mu_rho.cpu().numpy())

assert torch.allclose(mu_sparse, mu_vector)
assert mu_rho.shape == (2,)


mu from sparse ket : [0.+0.j 0.+0.j]
mu from state vector: [0.+0.j 0.+0.j]
mu from rho         : [0.4+0.j 0. +0.j]
